# TimesNet on Server - Setup and Training

This notebook contains all commands needed to run TimesNet on the remote server.

## Check GPU Availability

In [ ]:
!nvidia-smi

## Check Current Directory

In [ ]:
!pwd
!ls -la

## Navigate to Project Directory

In [ ]:
%cd /home/fzf/dev/TSAD/TimesNet/Time-Series-Library

## Setup Conda Environment (First Time Only)

Run these cells only once to create the environment:

In [ ]:
# Create conda environment (only run once)
!conda create -n timesnet python=3.9 -y

In [ ]:
# Register environment as Jupyter kernel (only run once)
!conda run -n timesnet pip install ipykernel
!conda run -n timesnet python -m ipykernel install --user --name=timesnet --display-name "Python (timesnet)"

## Install Dependencies

**IMPORTANT:** After running the cells above, switch the kernel to "Python (timesnet)" using the kernel selector in the top right!

Then run these cells to install packages:

In [2]:
# Verify we're using the correct environment
import sys
print(f"Python path: {sys.executable}")
print(f"Should contain 'timesnet' in the path")

Python path: /home/fzf/.conda/envs/timesnet/bin/python
Should contain 'timesnet' in the path


In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio

In [ ]:
# Install other dependencies
!pip install einops reformer-pytorch local-attention sktime sympy PyWavelets patool tqdm huggingface_hub datasets pandas numpy matplotlib scikit-learn

In [ ]:
# Install additional requirements if requirements.txt exists
!pip install -r requirements.txt

## Verify GPU is Available in Python

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

## Run TimesNet

Choose which GPU to use by setting CUDA_VISIBLE_DEVICES:

In [ ]:
# Set which GPU to use (0, 1, 2, etc.)
# IMPORTANT: This must run BEFORE any torch.cuda call (e.g. the "Verify GPU" cell).
# If CUDA is already initialized, restart the kernel first.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'  # Only expose physical GPU 3 → cuda:0

## SMD Dataset

In [40]:
!python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/SMD \
  --model_id SMD \
  --model TimesNet \
  --data SMD \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 2 \
  --enc_in 38 \
  --c_out 38 \
  --top_k 5 \
  --anomaly_ratio 0.5 \
  --batch_size 128 \
  --train_epochs 10

## MSL dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/MSL \
  --model_id MSL \
  --model TimesNet \
  --data MSL \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 8 \
  --d_ff 16 \
  --e_layers 1 \
  --enc_in 55 \
  --c_out 55 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 1

## SMAP Dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=3 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/SMAP \
  --model_id SMAP \
  --model TimesNet \
  --data SMAP \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 128 \
  --d_ff 128 \
  --e_layers 3 \
  --enc_in 25 \
  --c_out 25 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 3

## SWaT dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/SWaT \
  --model_id SWAT \
  --model TimesNet \
  --data SWAT \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 8 \
  --d_ff 8 \
  --e_layers 3 \
  --enc_in 51 \
  --c_out 51 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 3


## PSM dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/PSM \
  --model_id PSM \
  --model TimesNet \
  --data PSM \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 2 \
  --enc_in 25 \
  --c_out 25 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 3

## Run TimesNet on GECCO Dataset

### Prepare GECCO Dataset

In [1]:
!python scripts/prepare_gecco.py

Reading /home/fzf/dev/TSAD/datasets/GECCO/gecco2018_water_quality.csv...
Feature columns (9): ['Tp', 'Cl', 'pH', 'Redox', 'Leit', 'Trueb', 'Cl_2', 'Fm', 'Fm_2']
Label column: EVENT
Total samples: 139566, Features: 9
Anomaly ratio: 1.24%
Train samples: 83739, Test samples: 55827
Test anomaly ratio: 0.45%
Saved train.npy, test.npy, test_label.npy to /home/fzf/dev/TSAD/TimesNet/Time-Series-Library/dataset/GECCO

Done! Use --enc_in 9 --c_out 9 when running TimesNet.


In [ ]:
# Run TimesNet training on GECCO dataset
!python -u run.py \
  --task_name anomaly_detection \
  --is_training 1 \
  --root_path ./dataset/GECCO \
  --model_id GECCO \
  --model TimesNet \
  --data GECCO \
  --features M \
  --seq_len 10 \
  --pred_len 0 \
  --d_model 128 \
  --d_ff 128 \
  --e_layers 3 \
  --enc_in 9 \
  --c_out 9 \
  --top_k 5 \
  --anomaly_ratio 0.005 \
  --batch_size 128 \
  --train_epochs 3

### Neural Network Intelligence (NNI) toolkit

In [26]:
!nnictl create --config nni_config.yml --port 8081

/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/nni/tools/nnictl/nnictl.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-09 16:01:56] Creating experiment, Experiment ID: wca8nz6d
[2026-03-09 16:01:56] Starting web server...
[2026-03-09 16:01:57] Setting up...
[2026-03-09 16:01:57] Web portal URLs: http://127.0.0.1:8081 http://10.28.27.64:8081 http://172.17.0.1:8081
[2026-03-09 16:01:57] To stop experiment run "nnictl stop wca8nz6d" or "nnictl stop --all"
[2026-03-09 16:01:57] Reference: https://nni.readthedocs.io/en/stable/reference/nnictl.html


In [1]:
!nnictl stop --all

/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/nni/tools/nnictl/nnictl.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO:  Stopping experiment wca8nz6d
INFO:  Stop experiment success.


In [2]:
!lsof -i :8081

In [15]:
!kill -9 1953557

## Parameter Count

In [2]:
# ── Parameter count for TimesNet on ALL datasets (best hyperparams) ───
import sys, argparse
import torch

sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet

datasets = {
    'SMD':   dict(seq_len=100, enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
    'MSL':   dict(seq_len=100, enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
    'SMAP':  dict(seq_len=100, enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
    'SWaT':  dict(seq_len=100, enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
    'PSM':   dict(seq_len=100, enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
    'GECCO': dict(seq_len=10,  enc_in=9,  c_out=9,  d_model=128, d_ff=128, e_layers=3, top_k=5),
}

print(f"{'Dataset':<8} {'Total Params':>14} {'Trainable':>14} {'Non-trainable':>15} {'MB (fp32)':>10}")
print("-" * 67)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model = TimesNet(cfg).cpu().eval()

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    mb_fp32   = total * 4 / 1e6  # params * 4 bytes (float32)

    print(f"{name:<8} {total:>14,} {trainable:>14,} {frozen:>15,} {mb_fp32:>10.2f}")


Dataset    Total Params      Trainable   Non-trainable  MB (fp32)
-------------------------------------------------------------------
SMD           4,697,510      4,697,510               0      18.79
MSL              75,223         75,223               0       0.30
SMAP         28,133,145     28,133,145               0     112.53
SWaT            111,843        111,843               0       0.45
PSM           4,694,169      4,694,169               0      18.78
GECCO        28,124,937     28,124,937               0     112.50


In the results, SMD has ~4.7M params because it uses d_model=64, d_ff=64, e_layers=2 (larger hidden dims), while SWaT has only ~112K params because it uses d_model=8, d_ff=8 (much smaller). The parameter count determines how much memory the model needs and is independent of input sequence length — unlike MACs, which scale with seq_len.

## Number of Operations

In [14]:
# ── Count MACs for TimesNet on ALL datasets (best hyperparams) ─────────
import sys, argparse
import torch
import torch.nn as nn

sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet
from fvcore.nn import FlopCountAnalysis
from thop import profile

class _Wrap(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.inner = m
    def forward(self, x):
        return self.inner(x, None, None, None)

# Best hyperparams per dataset (from result_anomaly_detection.txt)
datasets = {
    # SMD:  F1=0.8459  dm64 df64 el2
    'SMD':   dict(seq_len=100,  enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
    # MSL:  F1=0.8180  dm8 df16 el1
    'MSL':   dict(seq_len=100,  enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
    # SMAP: F1=0.6944  dm128 df128 el3
    'SMAP':  dict(seq_len=100,  enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
    # SWaT: F1=0.9262  dm8 df8 el3
    'SWaT':  dict(seq_len=100,  enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
    # PSM:  F1=0.9738  dm64 df64 el2
    'PSM':   dict(seq_len=100,  enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
    # GECCO: F1=0.4504  sl10 dm128 df128 el3
    'GECCO': dict(seq_len=10,  enc_in=9,  c_out=9,  d_model=128,  d_ff=128,  e_layers=3, top_k=5),
}

print(f"{'Dataset':<8} {'fvcore MACs':>14} {'thop MACs':>14} {'Diff%':>10}")
print("-" * 50)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model = TimesNet(cfg).cpu().eval()
    wrapped = _Wrap(model)
    torch.manual_seed(42)
    dummy = torch.randn(1, cfg.seq_len, cfg.enc_in)

    with torch.no_grad():
        # fvcore
        fa = FlopCountAnalysis(wrapped, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        fvcore_macs = fa.total()

        # thop
        thop_macs, _ = profile(wrapped, inputs=(dummy,), verbose=False)

    diff_pct = abs(fvcore_macs - thop_macs) / max(fvcore_macs, 1) * 100
    print(f"{name:<8} {fvcore_macs:>14,} {thop_macs:>14,.0f} {diff_pct:>9.4f}%")

Dataset     fvcore MACs      thop MACs      Diff%
--------------------------------------------------
SMD       2,400,178,688  2,400,165,888    0.0005%
MSL          22,876,960     22,876,160    0.0035%
SMAP      8,773,334,528  8,773,296,128    0.0004%
SWaT         33,268,832     33,266,432    0.0072%
PSM       1,462,681,088  1,462,668,288    0.0009%
GECCO     1,462,042,368  1,462,038,528    0.0003%


### Normalizing for streaming data

In [ ]:
# ── Count MACs for TimesNet on ALL datasets (best hyperparams) ─────────
import sys, argparse
import torch
import torch.nn as nn
sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet
from fvcore.nn import FlopCountAnalysis
from thop import profile

class _Wrap(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.inner = m
    def forward(self, x):
        return self.inner(x, None, None, None)

datasets = {
    'SMD':   dict(seq_len=100, enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
    'MSL':   dict(seq_len=100, enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
    'SMAP':  dict(seq_len=100, enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
    'SWaT':  dict(seq_len=100, enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
    'PSM':   dict(seq_len=100, enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
    'GECCO': dict(seq_len=10,  enc_in=9,  c_out=9,  d_model=128, d_ff=128, e_layers=3, top_k=5),
}

print(
    f"{'Dataset':<8} {'seq_len':>8}  "
    f"{'fvcore/sample':>22} {'fvcore/t-point':>22}  "
    f"{'thop/sample':>22} {'thop/t-point':>22}"
)
print("-" * 115)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model   = TimesNet(cfg).cpu().eval()
    wrapped = _Wrap(model)
    torch.manual_seed(42)

    dummy = torch.randn(1, cfg.seq_len, cfg.enc_in)

    with torch.no_grad():
        fa = FlopCountAnalysis(wrapped, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        fvcore_sample = fa.total()

        thop_sample, _ = profile(wrapped, inputs=(dummy,), verbose=False)

    fvcore_timepoint = fvcore_sample / cfg.seq_len
    thop_timepoint   = thop_sample   / cfg.seq_len

    print(
        f"{name:<8} {cfg.seq_len:>8}  "
        f"{fvcore_sample:>22,} {int(fvcore_timepoint):>22,}  "
        f"{int(thop_sample):>22,} {int(thop_timepoint):>22,}"
    )

print()
print("Columns:")
print("  fvcore/sample  = MACs to process one full sliding window (fvcore)")
print("  fvcore/t-point = effective MACs per new arriving time point (fvcore)")
print("  thop/sample    = MACs to process one full sliding window (thop)")
print("  thop/t-point   = effective MACs per new arriving time point (thop)")
print("  t-point        = MACs/sample ÷ seq_len")

Dataset   seq_len           fvcore/sample         fvcore/t-point             thop/sample           thop/t-point
-------------------------------------------------------------------------------------------------------------------
SMD           100           2,400,178,688             24,001,786           2,400,165,888             24,001,658
MSL           100              22,876,960                228,769              22,876,160                228,761
SMAP          100           8,773,334,528             87,733,345           8,773,296,128             87,732,961
SWaT          100              33,268,832                332,688              33,266,432                332,664
PSM           100           1,462,681,088             14,626,810           1,462,668,288             14,626,682
GECCO          10           1,462,042,368            146,204,236           1,462,038,528            146,203,852

Columns:
  fvcore/sample  = MACs to process one full sliding window (fvcore)
  fvcore/t-point = eff

## Memory Usage

In [1]:
import torch
import torch.nn as nn
import argparse
import sys, os
sys.path.insert(0, "./")
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # change if needed

from exp.exp_anomaly_detection import Exp_Anomaly_Detection

# ── Force CUDA initialization ────────────────────────────────────────
_ = torch.zeros(1).cuda()
device = torch.device("cuda:0")
print(f"Using device: {torch.cuda.get_device_name(0)}")

# ── Dataset configurations (best hyperparams from your runs) ─────────
DATASETS = {
    "SMD": dict(
        model_id="SMD", data="SMD",
        root_path="./dataset/SMD",
        seq_len=100, enc_in=38, c_out=38,
        d_model=64, d_ff=64, e_layers=2, top_k=5,
        anomaly_ratio=0.5,
    ),
    "MSL": dict(
        model_id="MSL", data="MSL",
        root_path="./dataset/MSL",
        seq_len=100, enc_in=55, c_out=55,
        d_model=8, d_ff=16, e_layers=1, top_k=3,
        anomaly_ratio=1.0,
    ),
    "SMAP": dict(
        model_id="SMAP", data="SMAP",
        root_path="./dataset/SMAP",
        seq_len=100, enc_in=25, c_out=25,
        d_model=128, d_ff=128, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    "SWaT": dict(
        model_id="SWAT", data="SWAT",
        root_path="./dataset/SWaT",
        seq_len=100, enc_in=51, c_out=51,
        d_model=8, d_ff=8, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    "PSM": dict(
        model_id="PSM", data="PSM",
        root_path="./dataset/PSM",
        seq_len=100, enc_in=25, c_out=25,
        d_model=64, d_ff=64, e_layers=2, top_k=3,
        anomaly_ratio=1.0,
    ),
    "GECCO": dict(
        model_id="GECCO", data="GECCO",
        root_path="./dataset/GECCO",
        seq_len=10, enc_in=9, c_out=9,
        d_model=128, d_ff=128, e_layers=3, top_k=5,
        anomaly_ratio=0.005,
    ),
}

# ── Fixed args shared across all datasets ────────────────────────────
BASE_ARGS = dict(
    task_name="anomaly_detection", is_training=0,
    model="TimesNet", data_path="ETTh1.csv",
    features="M", target="OT", freq="h",
    checkpoints="./checkpoints/",
    label_len=48, pred_len=0,
    dec_in=7, n_heads=8, d_layers=1,
    moving_avg=25, factor=1, distil=True,
    dropout=0.1, embed="timeF", activation="gelu",
    num_kernels=6, batch_size=128, train_epochs=10,
    patience=3, learning_rate=0.0001, des="test",
    loss="MSE", lradj="type1", use_amp=False,
    num_workers=10, itr=1, use_gpu=True, gpu=0,
    gpu_type="cuda", use_multi_gpu=False,
    devices="0,1,2,3", p_hidden_dims=[128, 128],
    p_hidden_layers=2, expand=2, d_conv=4,
    output_attention=False,
)

# ── Measure memory for each dataset ──────────────────────────────────
results = {}

for dataset_name, ds_args in DATASETS.items():
    print(f"Measuring {dataset_name}...")
    try:
        # Build args
        args = argparse.Namespace(**{**BASE_ARGS, **ds_args})

        # Clear GPU memory from previous iteration
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        baseline = torch.cuda.memory_allocated(device)

        # Load model
        exp = Exp_Anomaly_Detection(args)
        model = exp.model
        weight_mem = (torch.cuda.memory_allocated(device) - baseline) / 1e6

        # Dummy input matching dataset dimensions
        dummy = torch.randn(
            args.batch_size, args.seq_len, args.enc_in
        ).to(device)
        criterion = nn.MSELoss()

        # Peak training memory (forward + backward)
        torch.cuda.reset_peak_memory_stats(device)
        output = model(dummy, None, None, None)
        loss = criterion(output, dummy)
        loss.backward()
        peak_train = torch.cuda.max_memory_allocated(device) / 1e6

        # Peak inference memory (no gradients)
        torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            output = model(dummy, None, None, None)
        peak_infer = torch.cuda.max_memory_allocated(device) / 1e6

        total_params = sum(p.numel() for p in model.parameters())

        results[dataset_name] = {
            "params":       total_params,
            "weight_exact": total_params * 4 / 1e6,  # params × 4 bytes → MB
            "weight_mb":    weight_mem,  # includes buffers, varies
            "train_mb":     peak_train,
            "infer_mb":     peak_infer,
            "overhead_mb":  peak_train - weight_mem,
        }

        # Clean up before next dataset
        del model, exp, dummy, output, loss
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ERROR: {e}")
        results[dataset_name] = None

# ── Print summary table ───────────────────────────────────────────────
print("\n" + "=" * 95)
print("MEMORY SUMMARY — TimesNet Anomaly Detection (batch_size=128)")
print("=" * 95)
print(f"{'Dataset':<8} {'Params':<12} {'Weights exact':<15} {'Weights GPU':<13} "
      f"{'Train peak':<13} {'Infer peak':<13} {'Overhead':<10}")
print(f"{'':>20} {'(params×4B)':>15} {'(allocated)':>13} "
      f"{'(MB)':>13} {'(MB)':>13} {'(MB)':>10}")
print("-" * 95)
for name, r in results.items():
    if r is None:
        print(f"{name:<8} ERROR")
    else:
        print(f"{name:<8} {r['params']:>10,}   {r['weight_exact']:>11.2f} MB  "
              f"{r['weight_mb']:>9.2f} MB  "
              f"{r['train_mb']:>9.2f} MB  "
              f"{r['infer_mb']:>9.2f} MB  "
              f"{r['overhead_mb']:>7.2f} MB")

/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: NVIDIA GeForce RTX 2080
Measuring SMD...
Use GPU: cuda:0
Measuring MSL...
Use GPU: cuda:0
Measuring SMAP...
Use GPU: cuda:0
Measuring SWaT...
Use GPU: cuda:0
Measuring PSM...
Use GPU: cuda:0
Measuring GECCO...
Use GPU: cuda:0

MEMORY SUMMARY — TimesNet Anomaly Detection (batch_size=128)
Dataset  Params       Weights exact   Weights GPU   Train peak    Infer peak    Overhead  
                         (params×4B)   (allocated)          (MB)          (MB)       (MB)
-----------------------------------------------------------------------------------------------
SMD       4,697,510         18.79 MB      20.08 MB     349.37 MB     275.58 MB   329.29 MB
MSL          75,223          0.30 MB       0.47 MB      70.51 MB      60.96 MB    70.04 MB
SMAP     28,133,145        112.53 MB     116.67 MB     877.91 MB     704.00 MB   761.24 MB
SWaT        111,843          0.45 MB       0.64 MB      71.63 MB      50.46 MB    70.99 MB
PSM       4,694,169         18.78 MB      20.77 MB     33